<a href="https://colab.research.google.com/github/lopitagabriela2627-ui/data_acquisition/blob/main/lopita_cantik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== 0. SETUP & FUNGSI BANTU =====
import os, re, io, json, time, gzip, hashlib, datetime as dt
import requests, pandas as pd, numpy as np
from urllib import robotparser
from urllib.parse import urlparse

BASE = "UTS-DAM"
for d in ["data/raw", "data/processed", "data/output", "docs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

KONTAK = "isi-email-kelompok@contoh.ac.id"   # <-- GANTI dengan email kelompok
UA_NAME = "UAS-DesaWisata-Mabar"
HEADERS = {"User-Agent": f"{UA_NAME}/1.0 (tugas kuliah; {KONTAK})"}
TODAY = dt.date.today().isoformat()

PROV = []       # catatan provenance
HTTP_LOG = []   # catatan status HTTP / retry (bukti Soal 3)

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def log_source(nama, url, path, lisensi, catatan=""):
    PROV.append({"nama": nama, "url": url, "tanggal_akses": TODAY, "lisensi": lisensi,
                 "file": path, "ukuran_byte": os.path.getsize(path), "md5": md5_file(path), "catatan": catatan})

def fetch(url, params=None, method="get", data=None, max_retry=5, timeout=90, label=""):
    """HTTP request dengan penanganan status & exponential backoff."""
    for i in range(1, max_retry + 1):
        try:
            r = requests.request(method, url, params=params, data=data, headers=HEADERS, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as e:
            HTTP_LOG.append({"label": label, "percobaan": i, "status": f"ERR {type(e).__name__}"})
            time.sleep(2 ** i); continue
        HTTP_LOG.append({"label": label, "percobaan": i, "status": r.status_code})
        if r.status_code == 200:
            return r
        if r.status_code in (429, 500, 502, 503, 504):      # sementara -> retry
            wait = int(r.headers["Retry-After"]) if r.headers.get("Retry-After", "").isdigit() else 2 ** i
            print(f"[{label}] HTTP {r.status_code} -> retry {i}/{max_retry} dalam {wait}s"); time.sleep(wait); continue
        r.raise_for_status()                                  # 4xx lain -> gagal langsung
    raise RuntimeError(f"Gagal setelah {max_retry} percobaan: {url}")

def norm(s):
    """Normalisasi nama wilayah untuk kunci merge."""
    s = str(s).lower()
    s = re.sub(r"\[[^\]]*\]", "", s)
    s = re.sub(r"\b(kecamatan|kelurahan|desa|kampung|kab\.?|kabupaten)\b", "", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
print("Siap. Tanggal akses:", TODAY)

Siap. Tanggal akses: 2026-11-26


In [ ]:
URL1_PAGE = "https://katalogdata.kemenpar.go.id/dataset/jumlah-desa-wisata-berkelanjutan-tersertifikasi-tahun-2025/resource/0119776f-6fbd-4961-ac5d-5039d4336b1a"
URL1_CSV  = "https://katalogdata.kemenpar.go.id/datastore/dump/0119776f-6fbd-4961-ac5d-5039d4336b1a?bom=True"

r = fetch(URL1_CSV, label="S1-CSV")
RAW1 = f"{BASE}/data/raw/kemenpar_desa_wisata_berkelanjutan_2025.csv"
open(RAW1, "wb").write(r.content)
log_source("Kemenpar - Desa Wisata Standar Pariwisata Berkelanjutan 2025", URL1_PAGE, RAW1,
           "Tidak dinyatakan pada portal (License not specified) - gunakan untuk tujuan akademik, cantumkan atribusi")

df1 = pd.read_csv(RAW1, encoding="utf-8-sig")
df1.columns = [c.strip().lower() for c in df1.columns]
print("Jumlah baris x kolom (nasional):", df1.shape)
print("Memenuhi >= 100 baris:", len(df1) >= 100)
print("\n--- Tipe data ---"); print(df1.dtypes)
print("\nUkuran file (KB):", round(os.path.getsize(RAW1) / 1024, 1),
      "| memori DataFrame (KB):", round(df1.memory_usage(deep=True).sum() / 1024, 1))
print("\n--- Cakupan wilayah ---")
for c in ["provinsi", "kabupaten_kota"]:
    if c in df1: print(c, "->", df1[c].nunique(), "nilai unik")
tc = "tahun_awal_sertifikasi"
if tc in df1: print("Cakupan waktu:", df1[tc].min(), "s.d.", df1[tc].max())
df1.head()

Jumlah baris x kolom (nasional): (10, 12)
Memenuhi >= 100 baris: False

--- Tipe data ---
_id                        int64
tahun_awal_sertifikasi     int64
desa_wisata               object
kode_desa_kelurahan        int64
desa_kelurahan            object
kode_kecamatan             int64
kecamatan                 object
kode_kabupaten_kota        int64
kabupaten_kota            object
kode_provinsi              int64
provinsi                  object
masa_berlaku              object
dtype: object

Ukuran file (KB): 1.4 | memori DataFrame (KB): 4.3

--- Cakupan wilayah ---
provinsi -> 7 nilai unik
kabupaten_kota -> 10 nilai unik
Cakupan waktu: 2025 s.d. 2025


,_id,tahun_awal_sertifikasi,desa_wisata,kode_desa_kelurahan,desa_kelurahan,kode_kecamatan,kecamatan,kode_kabupaten_kota,kabupaten_kota,kode_provinsi,provinsi,masa_berlaku
0,1,2025,Desa Wisata Wanurejo,3308020010,Desa Wanurejo,3308020,Kecamatan Borobudur,3371,Kabupaten Magelang,33,Jawa Tengah,2025-2028
1,2,2025,Desa Wisata Krebet,3402160002,Kelurahan Sendangsari,3402160,Kecamatan Pajangan,3402,Kabupaten Bantul,34,DI Yogyakarta,2025-2028
2,3,2025,Desa Wisata Les,5108090007,Desa Les,5108090,Kecamatan Tejakula,5108,Kabupaten Buleleng,51,Bali,2025-2028
3,4,2025,Desa Wisata Senaru,5208050005,Desa Senaru,5208050,Kecamatan Bayan,5208,Kabupaten Lombok Utara,52,Nusa Tenggara Barat,2025-2028
4,5,2025,Desa Wisata Pentingsari,3404170005,Desa Umbulharjo,3404170,Kecamatan Cangkringan,3404,Kabupaten Sleman,34,DI Yogyakarta,2025-2028


**filter tiga kabupaten dan estimasi penyimpanan**

In [ ]:
# Filter Manggarai Raya (pencocokan PERSIS agar "Manggarai" tidak ikut menangkap Barat/Timur)
KAB = {"manggarai barat": "Manggarai Barat", "manggarai": "Manggarai", "manggarai timur": "Manggarai Timur"}
df1["key_kab"] = df1["kabupaten_kota"].astype(str).map(norm)
df1_raya = df1[df1["key_kab"].isin(KAB)].copy()
df1_raya["kabupaten"] = df1_raya["key_kab"].map(KAB)
print("Baris Manggarai Raya:", len(df1_raya), "| >= 100 baris:", len(df1_raya) >= 100)
print(df1_raya["kabupaten"].value_counts().to_string())
display(df1_raya.head(10))

# Estimasi penyimpanan tahunan (asumsi dapat diubah)
size_kb = os.path.getsize(RAW1) / 1024
gz_kb = len(gzip.compress(open(RAW1, "rb").read())) / 1024
SNAPSHOT_PER_TAHUN = 1      # dirilis/diperbarui 1x per tahun (ganti 12 bila bulanan)
BUFFER = 1.2                # +20% metadata/log/versi
est = pd.DataFrame({
    "skenario": ["CSV mentah", "CSV terkompres gzip"],
    "per_snapshot_KB": [size_kb, gz_kb],
    "estimasi_per_tahun_KB": [size_kb * SNAPSHOT_PER_TAHUN * BUFFER, gz_kb * SNAPSHOT_PER_TAHUN * BUFFER]})
print(f"Rumus: ukuran_snapshot x {SNAPSHOT_PER_TAHUN} snapshot/tahun x buffer {BUFFER}")
est.round(2)

Baris Manggarai Raya: 0 | >= 100 baris: False
Series([], )


,_id,tahun_awal_sertifikasi,desa_wisata,kode_desa_kelurahan,desa_kelurahan,kode_kecamatan,kecamatan,kode_kabupaten_kota,kabupaten_kota,kode_provinsi,provinsi,masa_berlaku,key_kab,kabupaten


Rumus: ukuran_snapshot x 1 snapshot/tahun x buffer 1.2


,skenario,per_snapshot_KB,estimasi_per_tahun_KB
0,CSV mentah,1.44,1.73
1,CSV terkompres gzip,0.59,0.71


**pemindaian PII dan quasi-identifier**

In [ ]:
PII_PATTERNS = {
    "email":   r"[\w\.-]+@[\w\.-]+\.[a-z]{2,}",
    "telepon": r"(?:\+62|62|0)8\d{7,12}",
    "nik_16d": r"\b\d{16}\b",
}
def scan_pii(df, nama):
    hasil = []
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna().astype(str)
        for jenis, pat in PII_PATTERNS.items():
            n = int(s.str.contains(pat, regex=True, flags=re.I).sum())
            if n: hasil.append({"dataset": nama, "kolom": col, "jenis_pii": jenis, "jumlah_temuan": n})
    return pd.DataFrame(hasil, columns=["dataset", "kolom", "jenis_pii", "jumlah_temuan"])

pii_s1 = scan_pii(df1, "Kemenpar-CSV")
print("Temuan PII pada dataset Soal 1:", len(pii_s1)); display(pii_s1)

QUASI = [c for c in df1.columns if c in ("desa_wisata", "desa_kelurahan", "kecamatan", "kabupaten_kota", "provinsi", "tahun_awal_sertifikasi")]
print("Kandidat quasi-identifier:", QUASI)
print("Catatan: semua berupa data wilayah/entitas (bukan individu); risiko re-identifikasi rendah. Tidak ada nama orang/kontak pribadi.")

Temuan PII pada dataset Soal 1: 0


,dataset,kolom,jenis_pii,jumlah_temuan


Kandidat quasi-identifier: ['tahun_awal_sertifikasi', 'desa_wisata', 'desa_kelurahan', 'kecamatan', 'kabupaten_kota', 'provinsi']
Catatan: semua berupa data wilayah/entitas (bukan individu); risiko re-identifikasi rendah. Tidak ada nama orang/kontak pribadi.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


**ambil data via API Overpass (OpenStreetMap)**

In [ ]:
OVERPASS = "https://overpass-api.de/api/interpreter"
BBOX = {"Manggarai Barat": (-9.05, 119.65, -8.20, 120.50),
        "Manggarai": (-8.85, 120.15, -8.35, 120.65),
        "Manggarai Timur": (-8.90, 120.40, -8.25, 121.00)}

def q_area(nama):
    return ('[out:json][timeout:90];\n'
            'area["boundary"="administrative"]["name"="Kabupaten %s"]->.a;\n'
            '(node["place"~"^(village|hamlet|suburb|town|island)$"](area.a);node["tourism"](area.a););\n'
            'out tags center;') % nama
def q_bbox(b):
    s = "%s,%s,%s,%s" % b
    return ('[out:json][timeout:90];\n(node["place"~"^(village|hamlet|suburb|town|island)$"](%s);'
            'node["tourism"](%s););\nout tags center;') % (s, s)

elems = []
for nama, bb in BBOX.items():
    js = fetch(OVERPASS, method="post", data={"data": q_area(nama)}, label=f"S3-{nama}").json()
    metode = "area"
    if len(js.get("elements", [])) < 10:
        print(f"{nama}: hasil area sedikit -> fallback bounding box (kabupaten bisa kurang akurat)")
        js = fetch(OVERPASS, method="post", data={"data": q_bbox(bb)}, label=f"S3-{nama}-bbox").json()
        metode = "bbox"
    raw = f"{BASE}/data/raw/osm_overpass_{nama.lower().replace(' ', '_')}.json"
    json.dump(js, open(raw, "w", encoding="utf-8"), ensure_ascii=False)
    log_source(f"OpenStreetMap via Overpass - {nama}", OVERPASS, raw, "ODbL 1.0 (c) OpenStreetMap contributors", f"metode={metode}")
    for e in js["elements"]:
        e["kabupaten"] = nama; e["metode_wilayah"] = metode
    elems += js["elements"]
    print(nama, "->", len(js["elements"]), "elemen ("+metode+")")
    time.sleep(3)   # jeda sopan antar request

df3 = pd.json_normalize(elems, sep="_")
df3.columns = [c.replace("tags_", "") for c in df3.columns]
print("Total baris hasil normalisasi:", len(df3), "| kolom:", df3.shape[1], "| >= 30 baris:", len(df3) >= 30)
print("\nLog status HTTP:"); display(pd.DataFrame([h for h in HTTP_LOG if h["label"].startswith("S3")]))

[S3-Manggarai Barat] HTTP 502 -> retry 1/5 dalam 2s
[S3-Manggarai Barat] HTTP 502 -> retry 2/5 dalam 4s
Manggarai Barat: hasil area sedikit -> fallback bounding box (kabupaten bisa kurang akurat)
Manggarai Barat -> 715 elemen (bbox)
[S3-Manggarai] HTTP 502 -> retry 1/5 dalam 2s
[S3-Manggarai] HTTP 502 -> retry 2/5 dalam 4s
[S3-Manggarai] HTTP 502 -> retry 3/5 dalam 8s
[S3-Manggarai] HTTP 502 -> retry 4/5 dalam 16s
Manggarai: hasil area sedikit -> fallback bounding box (kabupaten bisa kurang akurat)
Manggarai -> 433 elemen (bbox)
Manggarai Timur: hasil area sedikit -> fallback bounding box (kabupaten bisa kurang akurat)
[S3-Manggarai Timur-bbox] HTTP 502 -> retry 1/5 dalam 2s
Manggarai Timur -> 382 elemen (bbox)
Total baris hasil normalisasi: 1530 | kolom: 84 | >= 30 baris: True

Log status HTTP:


,label,percobaan,status
0,S3-Manggarai Barat,1,502
1,S3-Manggarai Barat,2,502
2,S3-Manggarai Barat,3,200
3,S3-Manggarai Barat-bbox,1,200
4,S3-Manggarai,1,502
5,S3-Manggarai,2,502
6,S3-Manggarai,3,502
7,S3-Manggarai,4,502
8,S3-Manggarai,5,200
9,S3-Manggarai-bbox,1,200


**pembersihan data OSM**

In [ ]:
# Pembersihan -> data/processed. Kolom kontak/alamat (berpotensi PII) DIBUANG (minimisasi data).
pii_cols = [c for c in df3.columns if re.search(r"phone|email|contact|fax|whatsapp|addr", c, re.I)]
print("Kolom kontak/alamat dibuang:", pii_cols)
keep = [c for c in ["id", "lat", "lon", "name", "place", "tourism", "kabupaten", "metode_wilayah", "wikidata"] if c in df3.columns]
osm = df3.drop(columns=pii_cols, errors="ignore")[keep].copy()
osm = osm.rename(columns={"id": "osm_id", "name": "nama"})
osm = osm[osm["nama"].notna()].drop_duplicates(subset=["osm_id"]).reset_index(drop=True)
osm["kategori"] = np.where(osm["place"].notna(), "wilayah", "objek_wisata") if "place" in osm else "objek_wisata"
osm["key_kab"] = osm["kabupaten"].map(norm)
osm["key_desa"] = osm["key_kab"] + "|" + osm["nama"].map(norm)
PROC3 = f"{BASE}/data/processed/osm_manggarai_raya_bersih.csv"
osm.to_csv(PROC3, index=False, encoding="utf-8")
print("Baris bersih:", len(osm)); print(osm["kabupaten"].value_counts().to_string()); osm.head(10)

Kolom kontak/alamat dibuang: ['email', 'phone', 'addr:city', 'addr:postcode', 'addr:street', 'addr:housenumber', 'addr:country', 'contact:language', 'addr:district', 'addr:housename', 'addr:neighbourhood', 'addr:province', 'addr:subdistrict', 'addr:unit']
Baris bersih: 910
kabupaten
Manggarai Barat    668
Manggarai Timur    158
Manggarai           84


,osm_id,lat,lon,nama,place,tourism,kabupaten,metode_wilayah,wikidata,kategori,key_kab,key_desa
0,703787171,-8.482560,119.880461,Golo Hilltop,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|golo hilltop
1,727766254,-8.492503,119.877713,Gardena,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|gardena
2,834902092,-8.508675,119.877342,New Bajo,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|new bajo
3,931543319,-8.582300,119.800000,Pulau Purung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|pulau purung
4,931543800,-8.624800,119.806200,Pulau Gadung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|pulau gadung
5,1070187399,-8.467900,119.878000,Bukit Binongko,NaN,viewpoint,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|bukit binongko
6,1118552623,-8.611507,120.445643,Compang Ruteng,hamlet,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|compang ruteng
7,1118552624,-8.611880,120.446034,Compang Ruteng,NaN,yes,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|compang ruteng
8,1271854768,-8.606215,120.347281,Pong Leko,village,NaN,Manggarai Barat,bbox,Q24639006,wilayah,manggarai barat,manggarai barat|pong leko
9,1271854770,-8.457569,120.341388,Lumut,village,NaN,Manggarai Barat,bbox,Q24597086,wilayah,manggarai barat,manggarai barat|lumut


**cek robots.txt dan scraping tiga halaman Wikipedia**

In [ ]:
PAGES4 = {"Manggarai Barat": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Barat",
          "Manggarai": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai",
          "Manggarai Timur": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Timur"}
ROBOTS4 = "https://id.wikipedia.org/robots.txt"

rob = fetch(ROBOTS4, label="S4-robots")
RAW4R = f"{BASE}/data/raw/robots_id_wikipedia.txt"
open(RAW4R, "w", encoding="utf-8").write(rob.text)
log_source("robots.txt id.wikipedia.org", ROBOTS4, RAW4R, "Bukti izin crawling")
rp = robotparser.RobotFileParser(); rp.parse(rob.text.splitlines())

TABS, CANDS = {}, {}
for nama, url in PAGES4.items():
    boleh = rp.can_fetch(UA_NAME, url)
    print(f"{nama}: diizinkan scraping -> {boleh}")
    assert boleh, f"robots.txt melarang {url} - JANGAN scraping"
    time.sleep(2)
    pg = fetch(url, label=f"S4-{nama}")
    raw = f"{BASE}/data/raw/wikipedia_kab_{nama.lower().replace(' ', '_')}.html"
    open(raw, "w", encoding="utf-8").write(pg.text)
    log_source(f"Wikipedia ID - Kabupaten {nama}", url, raw, "CC BY-SA 4.0 (Wikipedia)", "Scraping statis; atribusi wajib")
    TABS[nama] = pd.read_html(io.StringIO(pg.text))
    CANDS[nama] = [i for i, t in enumerate(TABS[nama]) if any("kecamatan" in str(c).lower() for c in t.columns) and len(t) >= 5]
    print("   tabel kandidat:", [(i, TABS[nama][i].shape) for i in CANDS[nama]])

Manggarai Barat: diizinkan scraping -> True
   tabel kandidat: [(5, (16, 6))]
Manggarai: diizinkan scraping -> True
   tabel kandidat: [(5, (16, 6))]
Manggarai Timur: diizinkan scraping -> True
   tabel kandidat: [(6, (17, 6))]


**parsing tabel kecamatan**

In [ ]:
TABLE_INDEX = {}    # isi manual bila tabel otomatis salah, contoh: {"Manggarai": 3}
parts = []
for nama in PAGES4:
    idx = TABLE_INDEX.get(nama, CANDS[nama][0] if CANDS[nama] else None)
    assert idx is not None, f"Tabel kecamatan {nama} tidak ditemukan - cek struktur halaman / isi TABLE_INDEX"
    t = TABS[nama][idx].copy()
    if isinstance(t.columns, pd.MultiIndex):
        t.columns = [" ".join(dict.fromkeys(str(x) for x in col)).strip() for col in t.columns]
    t.columns = [re.sub(r"\s+", "_", re.sub(r"\[[^\]]*\]", "", str(c)).strip().lower()) for c in t.columns]
    kcol = [c for c in t.columns if "kecamatan" in c][0]
    t = t.rename(columns={kcol: "kecamatan"})
    t = t[t["kecamatan"].notna() & ~t["kecamatan"].astype(str).str.lower().str.contains("total|jumlah")].copy()
    t["kecamatan"] = t["kecamatan"].astype(str).str.replace(r"\[[^\]]*\]", "", regex=True).str.strip()
    t["kabupaten"] = nama
    parts.append(t)
t4 = pd.concat(parts, ignore_index=True)
t4["key_kab"] = t4["kabupaten"].map(norm)
t4["key_kecamatan"] = t4["key_kab"] + "|" + t4["kecamatan"].map(norm)
t4 = t4.drop_duplicates(subset=["key_kecamatan"]).reset_index(drop=True)
PROC4 = f"{BASE}/data/processed/wikipedia_kecamatan_manggarai_raya.csv"
t4.to_csv(PROC4, index=False, encoding="utf-8")
print("Baris kecamatan:", len(t4)); print(t4["kabupaten"].value_counts().to_string()); t4.head(10)

Baris kecamatan: 34
kabupaten
Manggarai Barat    12
Manggarai          12
Manggarai Timur    10


,kode_kemendagri,kecamatan,jumlah_kelurahan,jumlah_desa,status,daftar_desa/kelurahan,kabupaten,key_kab,key_kecamatan
0,53.15.01,Macang Pacar,NaN,13,Desa,Bari Lewat Mbakung Nanga Kantor Nanga Kantor B...,Manggarai Barat,manggarai barat,manggarai barat|macang pacar
1,53.15.02,Kuwus,2.0,10,Desa,Bangka Lewat Benteng Suru Coal Compang Suka Go...,Manggarai Barat,manggarai barat,manggarai barat|kuwus
2,53.15.03,Lembor,1.0,14,Desa,Daleng Golo Ndeweng Liang Sola Ngancar Poco De...,Manggarai Barat,manggarai barat,manggarai barat|lembor
3,53.15.04,Sano Nggoang,NaN,15,Desa,Golo Kempo Golo Kondeng Golo Leleng Golo Manti...,Manggarai Barat,manggarai barat,manggarai barat|sano nggoang
4,53.15.05,Komodo,2.0,17,Desa,Batu Cermin Compang Longgo Golo Bilas Golo Mor...,Manggarai Barat,manggarai barat,manggarai barat|komodo
5,53.15.06,Boleng,NaN,11,Desa,Batu Tiga Beo Sepang Golo Ketak Golo Lujang Go...,Manggarai Barat,manggarai barat,manggarai barat|boleng
6,53.15.07,Welak,NaN,16,Desa,Dunta Galang Golo Ndari Golo Ronggot Gurung La...,Manggarai Barat,manggarai barat,manggarai barat|welak
7,53.15.08,Ndoso,NaN,15,Desa,Golo Bore Golo Keli Golo Poleng Golo Ru'a Lumu...,Manggarai Barat,manggarai barat,manggarai barat|ndoso
8,53.15.09,Lembor Selatan,NaN,15,Desa,Benteng Dewa Benteng Tado Kakor Lalong Lendong...,Manggarai Barat,manggarai barat,manggarai barat|lembor selatan
9,53.15.10,Mbeliling,NaN,15,Desa,Compang Liang Ndara Cunca Lolos Cunca Wulang G...,Manggarai Barat,manggarai barat,manggarai barat|mbeliling


integrasi (merge + validate + cek duplikasi)

In [ ]:
A = df1_raya.copy()
A["key_kecamatan"] = A["key_kab"] + "|" + A["kecamatan"].map(norm)
A["key_desa"] = A["key_kab"] + "|" + A["desa_kelurahan"].map(norm)
A = A.drop_duplicates(subset=["key_desa", "key_kecamatan"])
A.to_csv(f"{BASE}/data/processed/kemenpar_manggarai_raya_bersih.csv", index=False, encoding="utf-8")

# Merge 1: A x C (banyak desa -> satu kecamatan)
AC = A.merge(t4, on="key_kecamatan", how="left", validate="m:1", suffixes=("", "_wiki"), indicator="_m_kec")
print("Merge A x C:\n", AC["_m_kec"].value_counts().to_string())
AC = AC.drop(columns="_m_kec")

# Merge 2: B (OSM) x AC
AC_u = AC.drop_duplicates(subset=["key_desa"])
final = osm.merge(AC_u, on="key_desa", how="left", validate="m:1", suffixes=("", "_kemenpar"), indicator="_m_desa")
final["status_desa_wisata_kemenpar"] = np.where(final["_m_desa"] == "both", "tersertifikasi", "tidak_tercatat")
print("\nMerge OSM x (A+C):\n", final["_m_desa"].value_counts().to_string())
final = final.drop(columns="_m_desa")

print("\nDuplikat baris penuh:", int(final.duplicated().sum()), "| duplikat osm_id:", int(final["osm_id"].duplicated().sum()))
final = final.drop_duplicates().reset_index(drop=True)
OUT = f"{BASE}/data/output/desa_wisata_manggarai_raya_terintegrasi.csv"
final.to_csv(OUT, index=False, encoding="utf-8")
print("Hasil akhir:", final.shape); print(final.groupby(["kabupaten", "status_desa_wisata_kemenpar"]).size().to_string()); final.head()

Merge A x C:
 _m_kec
left_only     0
right_only    0
both          0

Merge OSM x (A+C):
 _m_desa
left_only     910
right_only      0
both            0

Duplikat baris penuh: 0 | duplikat osm_id: 0
Hasil akhir: (910, 36)
kabupaten        status_desa_wisata_kemenpar
Manggarai        tidak_tercatat                  84
Manggarai Barat  tidak_tercatat                 668
Manggarai Timur  tidak_tercatat                 158


,osm_id,lat,lon,nama,place,tourism,kabupaten,metode_wilayah,wikidata,kategori,...,key_kecamatan,kode_kemendagri,kecamatan_wiki,jumlah_kelurahan,jumlah_desa,status,daftar_desa/kelurahan,kabupaten_wiki,key_kab_wiki,status_desa_wisata_kemenpar
0,703787171,-8.482560,119.880461,Golo Hilltop,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
1,727766254,-8.492503,119.877713,Gardena,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
2,834902092,-8.508675,119.877342,New Bajo,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
3,931543319,-8.582300,119.800000,Pulau Purung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
4,931543800,-8.624800,119.806200,Pulau Gadung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat


**metadata dan provenance**

In [ ]:
# Metadata + dokumentasi provenance + checklist etika + README
meta = {
    "judul": "Dataset Terintegrasi Desa Wisata Manggarai Raya (Manggarai, Manggarai Timur, Manggarai Barat)",
    "dibuat": TODAY, "jumlah_baris": int(len(final)), "jumlah_kolom": int(final.shape[1]),
    "kolom": list(final.columns), "kunci_merge": ["key_kecamatan", "key_desa"],
    "sumber": [{k: p[k] for k in ("nama", "url", "lisensi", "md5")} for p in PROV],
    "md5_output": md5_file(OUT),
}
json.dump(meta, open(f"{BASE}/data/output/metadata.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
log_source("Hasil integrasi (output akhir)", "(dihasilkan notebook ini)", OUT, "Mengikuti lisensi sumber (ODbL & CC BY-SA -> share-alike)")

prov = pd.DataFrame(PROV)
with open(f"{BASE}/docs/provenance.md", "w", encoding="utf-8") as f:
    f.write("# Provenance Data\n\n| Sumber | URL | Tanggal akses | Lisensi | File | MD5 |\n|---|---|---|---|---|---|\n")
    for p in PROV:
        f.write(f"| {p['nama']} | {p['url']} | {p['tanggal_akses']} | {p['lisensi']} | `{p['file']}` | `{p['md5']}` |\n")
prov[["nama", "tanggal_akses", "md5"]]

,nama,tanggal_akses,md5
0,Kemenpar - Desa Wisata Standar Pariwisata Berk...,2026-11-26,9785f623e4122b8c863f0540c16cbb3b
1,OpenStreetMap via Overpass - Manggarai Barat,2026-11-26,6d92a9072f161b601b209c5aa5ee7839
2,OpenStreetMap via Overpass - Manggarai,2026-11-26,fc4b81ce2ed8a39a786bddf9751c848b
3,OpenStreetMap via Overpass - Manggarai Timur,2026-11-26,d4756e25db72be6242551ff71dc1cd49
4,robots.txt id.wikipedia.org,2026-11-26,8dd94bf3d01cbb25e0af4e64c0485433
5,Wikipedia ID - Kabupaten Manggarai Barat,2026-11-26,b2980601973a57de977e902e224c4cf6
6,Wikipedia ID - Kabupaten Manggarai,2026-11-26,875cdc6f2190ef256923502be98760e9
7,Wikipedia ID - Kabupaten Manggarai Timur,2026-11-26,6cce511cdb5fc185f68c51a088b39787
8,Hasil integrasi (output akhir),2026-11-26,3a8a12df34af1e9fc87b6875a33f0d5b


**Data Ethics Checklist 9 poin**

In [ ]:
pii_all = pd.concat([pii_s1, scan_pii(final, "Output-akhir")], ignore_index=True)
temuan = "tidak ada temuan" if pii_all.empty else f"{len(pii_all)} temuan - periksa tabel di atas"
checklist = [
 ("1. Sumber & kepemilikan data jelas", "Kemenpar (pemerintah), OpenStreetMap contributors, Wikipedia (3 artikel kabupaten) - URL ada di docs/provenance.md"),
 ("2. Lisensi diperiksa", "Kemenpar: lisensi tidak dinyatakan pada portal (tangkapan halaman, tgl akses " + TODAY + "); OSM: ODbL 1.0; Wikipedia: CC BY-SA 4.0"),
 ("3. Ketentuan portal/ToS dipatuhi", "Overpass: 3 request kecil + jeda + retry/backoff; Wikipedia: robots.txt dicek (file raw tersimpan), User-Agent jelas"),
 ("4. Pemeriksaan data pribadi (PII)", "Pemindaian regex email/telepon/NIK: " + temuan + "; kolom kontak/alamat OSM dibuang"),
 ("5. Quasi-identifier dinilai", "Kolom wilayah/tahun saja: " + ", ".join(QUASI) + " - level entitas wilayah, bukan individu"),
 ("6. Minimisasi data", "Hanya kolom yang dibutuhkan disimpan pada data/processed"),
 ("7. Penyimpanan & integritas", "MD5 setiap file raw dicatat di docs/provenance.md; raw tidak diubah"),
 ("8. Atribusi & transparansi", "Atribusi sumber di README; hasil gabungan mengikuti ODbL/CC BY-SA (share-alike)"),
 ("9. Dampak & penggunaan wajar", "Untuk tujuan akademik; tidak ada profiling individu; scraping 3 halaman dengan jeda 2 detik"),
]
with open(f"{BASE}/docs/data_ethics_checklist.md", "w", encoding="utf-8") as f:
    f.write("# Data Ethics Checklist\n\n| Butir | Bukti |\n|---|---|\n")
    for a, b in checklist: f.write(f"| {a} | {b} |\n")
pd.DataFrame(checklist, columns=["Butir", "Bukti"])

,Butir,Bukti
0,1. Sumber & kepemilikan data jelas,"Kemenpar (pemerintah), OpenStreetMap contribut..."
1,2. Lisensi diperiksa,Kemenpar: lisensi tidak dinyatakan pada portal...
2,3. Ketentuan portal/ToS dipatuhi,Overpass: 3 request kecil + jeda + retry/backo...
3,4. Pemeriksaan data pribadi (PII),Pemindaian regex email/telepon/NIK: tidak ada ...
4,5. Quasi-identifier dinilai,Kolom wilayah/tahun saja: tahun_awal_sertifika...
5,6. Minimisasi data,Hanya kolom yang dibutuhkan disimpan pada data...
6,7. Penyimpanan & integritas,MD5 setiap file raw dicatat di docs/provenance...
7,8. Atribusi & transparansi,Atribusi sumber di README; hasil gabungan meng...
8,9. Dampak & penggunaan wajar,Untuk tujuan akademik; tidak ada profiling ind...


**README dengan link sumber dan arsip zip**
- *Nama*: GABRIELA ARNAIZ LOPITA
- *NIM*: 2501010326
- *Email*: 2501010326@instiki.ac.id
- *Mata Kuliah*:
- *Sektor*:

# Proyek Akuisisi Data Desa Wisata - Manggarai Raya

UAS Kualifikasi: akuisisi dan pengelolaan data berbasis **dataset terbuka, API, dan web scraping**.
Cakupan wilayah: Kabupaten Manggarai Barat, Kabupaten Manggarai, dan Kabupaten Manggarai Timur (NTT).

## 1. Tujuan
Mengumpulkan, memeriksa etikanya, membersihkan, lalu menggabungkan data desa wisata dari beberapa sumber terbuka menjadi satu dataset terintegrasi, lengkap dengan dokumentasi asal-usul data (provenance).

## 2. Sumber data
| Soal | Sumber | Link | Lisensi |
|---|---|---|---|
| 1 Dataset | Satu Data Kemenpar - Desa Wisata Standar Pariwisata Berkelanjutan 2025 | https://katalogdata.kemenpar.go.id/dataset/jumlah-desa-wisata-berkelanjutan-tersertifikasi-tahun-2025/resource/0119776f-6fbd-4961-ac5d-5039d4336b1a | tidak dinyatakan pada portal |
| 3 API | OpenStreetMap via Overpass API (query Manggarai Barat, Manggarai, Manggarai Timur) | https://overpass-api.de/api/interpreter | ODbL 1.0 |
| 4 Scraping | Wikipedia ID - Kabupaten Manggarai Barat | https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Barat | CC BY-SA 4.0 |
| 4 Scraping | Wikipedia ID - Kabupaten Manggarai | https://id.wikipedia.org/wiki/Kabupaten_Manggarai | CC BY-SA 4.0 |
| 4 Scraping | Wikipedia ID - Kabupaten Manggarai Timur | https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Timur | CC BY-SA 4.0 |
| 5 Integrasi | Gabungan semua sumber | `data/output/desa_wisata_manggarai_raya_terintegrasi.csv` | share-alike (ODbL / CC BY-SA) |

## 3. Penjelasan alur kerja notebook

### Soal 1 - Dataset terbuka
- Mengunduh CSV dari portal Satu Data Kemenpar lewat fungsi `fetch()` (dengan retry), menyimpannya apa adanya di `data/raw/`, dan mencatat MD5-nya.
- Memeriksa struktur: jumlah baris dan kolom, tipe data tiap kolom, ukuran file (KB), serta cakupan wilayah (provinsi, kabupaten/kota) dan waktu (`tahun_awal_sertifikasi`).
- Memeriksa syarat minimal **100 baris** pada dataset nasional, lalu menyaring tiga kabupaten Manggarai Raya dengan pencocokan nama persis (agar "Manggarai" tidak ikut menangkap "Manggarai Barat" dan "Manggarai Timur").
- **Estimasi penyimpanan tahunan** = ukuran file × jumlah snapshot per tahun × buffer 1,2 (cadangan 20% untuk metadata dan versi). Dihitung untuk CSV mentah dan CSV terkompres gzip. Asumsi awal: 1 snapshot per tahun; ubah `SNAPSHOT_PER_TAHUN` bila diperbarui lebih sering.

### Soal 2 - Etika data
- **Lisensi dan ketentuan portal** dicatat per sumber (tabel di atas). Portal Kemenpar tidak menyatakan lisensi, sehingga data dipakai untuk tujuan akademik dengan atribusi.
- **Pemindaian PII**: regex untuk email, nomor telepon Indonesia, dan NIK 16 digit pada semua kolom teks.
- **Quasi-identifier**: kolom desa, kecamatan, kabupaten, provinsi, dan tahun sertifikasi dinilai. Semuanya berupa data wilayah/entitas, bukan individu, sehingga risiko re-identifikasi rendah.
- **Minimisasi data**: kolom kontak/alamat dari OSM (phone, email, contact, fax, whatsapp, addr) dibuang sebelum disimpan.
- **Data Ethics Checklist 9 poin** dihasilkan otomatis ke `docs/data_ethics_checklist.md`, lengkap dengan bukti tiap butir.

### Soal 3 - API (Overpass / OpenStreetMap)
- Satu query per kabupaten untuk titik permukiman (`village`, `hamlet`, `suburb`, `town`, `island`) dan objek wisata (`tourism`).
- Penanganan status HTTP: 200 diterima; 429 dan 5xx di-retry dengan exponential backoff (maksimal 5 kali); error 4xx lain langsung gagal. Setiap percobaan dicatat di `HTTP_LOG` sebagai bukti.
- Jika query area mengembalikan kurang dari 10 elemen, notebook beralih ke bounding box kasar (kolom `metode_wilayah` menandai `area` atau `bbox`).
- JSON dinormalisasi dengan `pd.json_normalize` menjadi tabel dan diperiksa terhadap syarat minimal **30 baris**.

### Soal 4 - Web scraping
- `robots.txt` Wikipedia ID diunduh dan disimpan sebagai bukti, lalu `can_fetch()` dicek untuk tiap halaman. Jika tidak diizinkan, proses berhenti.
- Tiga halaman kabupaten diambil dengan jeda 2 detik, disimpan sebagai HTML mentah beserta MD5.
- Tabel kecamatan dipilih otomatis (tabel pertama yang memiliki kolom "kecamatan" dan minimal 5 baris), dibersihkan, lalu digabung menjadi satu tabel. Bila tabel yang terpilih salah, isi `TABLE_INDEX` secara manual.

### Soal 5 - Integrasi
Kunci merge selalu memuat kabupaten (`kabupaten|nama`) supaya nama yang sama di kabupaten berbeda tidak tertukar.
1. Kemenpar × Wikipedia pada `key_kecamatan` dengan `validate="m:1"` (banyak desa untuk satu kecamatan).
2. OSM × hasil langkah 1 pada `key_desa` dengan `validate="m:1"` (setelah deduplikasi).
3. Pengecekan duplikasi (baris penuh dan `osm_id`), lalu duplikat dibuang.
4. Kolom `status_desa_wisata_kemenpar` bernilai `tersertifikasi` bila desa OSM cocok dengan data Kemenpar, selain itu `tidak_tercatat`.
5. Hasil akhir dan `metadata.json` (jumlah baris/kolom, daftar kolom, kunci merge, sumber, MD5) disimpan.

## 4. Struktur keluaran
```
UTS-DAM/
├── notebook.ipynb
├── data/
│   ├── raw/        # data asli (CSV Kemenpar, JSON Overpass x3, HTML Wikipedia x3, robots.txt)
│   ├── processed/  # data bersih (Kemenpar, OSM, kecamatan Wikipedia)
│   └── output/     # desa_wisata_manggarai_raya_terintegrasi.csv + metadata.json
└── docs/
    ├── provenance.md             # URL, tanggal akses, lisensi, MD5
    └── data_ethics_checklist.md  # 9 poin checklist beserta bukti
```

## 5. Jumlah record data
Nilai diisi dari output sel di Google Colab setelah notebook dijalankan.

| Tahap | Sumber / sel | Jumlah record | Syarat | Terpenuhi? |
|---|---|---|---|---|
| Soal 1 - dataset nasional Kemenpar (baris x kolom) | Sel Soal 1, `Jumlah baris x kolom (nasional)` | | minimal 100 baris | |
| Soal 1 - difilter Manggarai Raya | `Baris Manggarai Raya` | | - | - |
| - Manggarai Barat | rincian per kabupaten | | - | - |
| - Manggarai | rincian per kabupaten | | - | - |
| - Manggarai Timur | rincian per kabupaten | | - | - |
| Soal 3 - API Overpass (hasil normalisasi) | `Total baris hasil normalisasi` | | minimal 30 baris | |
| Soal 3 - data OSM setelah dibersihkan | `Baris bersih` | | - | - |
| Soal 4 - kecamatan hasil scraping Wikipedia | `Baris kecamatan` | | - | - |
| Soal 5 - merge Kemenpar x Wikipedia | `Merge A x C` (`both` / `left_only`) | | - | - |
| Soal 5 - merge OSM x (Kemenpar + Wikipedia) | `Merge OSM x (A+C)` | | - | - |
| Soal 5 - duplikat ditemukan | `Duplikat baris penuh` / `duplikat osm_id` | | - | - |
| Soal 5 - dataset akhir (baris x kolom) | `Hasil akhir` | | - | - |

Informasi tambahan yang juga dicatat dari output:

| Item | Nilai |
|---|---|
| Ukuran file CSV (KB) dan estimasi per tahun (KB) | |
| Temuan PII | |
| Metode wilayah OSM per kabupaten (`area` / `bbox`) | |
| MD5 dataset akhir | |

## 6. Keterbatasan
- Dataset Kemenpar bersifat nasional dan hanya memuat desa wisata yang tersertifikasi, sehingga jumlah baris Manggarai Raya bisa jauh lebih kecil daripada total nasional.
- Pencocokan desa antara OSM dan Kemenpar memakai nama yang dinormalisasi. Perbedaan penulisan membuat sebagian desa tidak cocok (`tidak_tercatat`), dan itu bukan berarti desa tersebut bukan desa wisata.
- Bila Overpass memakai bounding box, batas kabupaten kasar dan saling tumpang tindih sehingga label kabupaten kurang akurat.
- Data OSM dan Wikipedia disunting sukarelawan, jadi kelengkapannya bervariasi.
- Lisensi Kemenpar tidak dinyatakan; gunakan hanya untuk keperluan akademik dengan atribusi.
